# 03. CodeExec Strategy & Container Execution

ArcRun ships two execution strategies. **ReAct** is the workhorse —
the model reasons, calls a predefined tool, observes, and repeats.
**CodeExec** flips the loop on its head: instead of calling tools, the
model writes Python and the runtime executes it in a sandbox.

This notebook walks the full surface — the strategy, the two
execution backends (local subprocess and Docker container), and the
prompt-provider API that exposes guidance to consuming agents.

**You will learn:**
- When CodeExec wins over ReAct (and when it loses)
- How `CodeExecStrategy` augments the system prompt before delegating
  to `react_loop`
- `make_execute_tool` — sandboxed Python via subprocess, isolated env
- `make_contained_execute_tool` — Docker isolation with `mem_limit`,
  `cpu_quota`, `pids_limit`, network disabled, read-only FS
- `prompt_guidance` property — what each strategy advertises about
  itself to the LLM
- `get_strategy_prompts()` — assembles model-facing guidance for
  multi-strategy mode
- How `select_strategy` decides which strategy runs when multiple
  are registered
- Mid-run task completion via `task_complete` (cross-ref
  `arcrun/06-task-completion-budgets.ipynb`)

Everything runs end-to-end without an API key — `MockModel` returns
the code blocks an LLM would generate. The container demo gates on
Docker availability and degrades cleanly when the daemon is not
running.

## 1. Setup

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Sibling-package conftest collisions can confuse `pytest` import
discovery, so we define `MockModel` (and the message dataclasses
the `arcllm` types module would normally provide) inline. This
keeps the notebook self-contained and avoids any cross-package
fixture leakage when the smoke run executes.

In [ ]:
from dataclasses import dataclass, field
from typing import Any


@dataclass
class Usage:
    input_tokens: int = 10
    output_tokens: int = 5
    total_tokens: int = 15


@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict[str, Any]


@dataclass
class TextBlock:
    text: str


@dataclass
class ToolUseBlock:
    id: str
    name: str
    arguments: dict[str, Any]


@dataclass
class ToolResultBlock:
    tool_use_id: str
    content: str


@dataclass
class Message:
    role: str
    content: Any


@dataclass
class LLMResponse:
    content: str | None = None
    tool_calls: list[ToolCall] = field(default_factory=list)
    stop_reason: str = "end_turn"
    usage: Usage = field(default_factory=Usage)
    cost_usd: float = 0.001


class MockModel:
    """Deterministic mock LLM. Records every invoke() call."""

    def __init__(self, responses: list[LLMResponse]) -> None:
        self._responses = list(responses)
        self._idx = 0
        self.invoke_calls: list[dict] = []

    async def invoke(self, messages, tools=None, **_kwargs) -> LLMResponse:
        self.invoke_calls.append({"messages": messages, "tools": tools})
        if self._idx >= len(self._responses):
            raise RuntimeError("MockModel exhausted responses")
        resp = self._responses[self._idx]
        self._idx += 1
        return resp


print("MockModel ready")

## 2. CodeExec vs ReAct — when each is the right tool

Both strategies use the same underlying engine (`react_loop`). The
difference is what the model is encouraged to do.

| | ReAct | CodeExec |
|---|---|---|
| **Model output** | Tool calls (predefined surface) | Python scripts |
| **Loop shape** | Reason → call tool → observe → repeat | Reason → write code → run code → observe → repeat |
| **Best for** | Multi-step problems with discrete tool actions (file ops, API calls, search) | Computation, data transformation, parsing, math |
| **Output you read back** | Tool result strings | `{stdout, stderr, exit_code, duration_ms}` |
| **State persistence** | Across turns (registry + workspace) | None — each script is stateless |
| **Sandbox** | Tool allowlist + per-tool checks | Subprocess or Docker container |

When in doubt: if the problem reduces to *"do this calculation and
report it"*, CodeExec wins. If the problem is *"orchestrate these
discrete operations against external systems"*, ReAct wins. Many
real agents register both and let the model pick (section 8).

In [ ]:
from arcrun import Strategy, get_strategy_prompts, make_execute_tool
from arcrun.strategies import STRATEGIES, _load_strategies, select_strategy
from arcrun.strategies.code import CodeExecStrategy, _DEFAULT_PREFIX
from arcrun.strategies.react import ReactStrategy

_load_strategies()

for name, strat in STRATEGIES.items():
    print(f"{name:>6s}  {type(strat).__name__:20s}  is Strategy: {isinstance(strat, Strategy)}")

## 3. The code strategy — composing turns

`CodeExecStrategy.__call__` is intentionally tiny. Its only job is
to **prepend a code-execution prompt** to the existing system
message before handing off to `react_loop`. The augmented prompt
tells the LLM:

1. You have an `execute_python` tool — write code, don't call tools
2. Each script gets `{stdout, stderr, exit_code, duration_ms}` back
3. Scripts are stateless across calls
4. Three-strikes rule: after three failures, change approach

An event (`code.prompt.augmented`) is emitted so the audit trail
captures the prompt mutation. Here is the canonical prefix the
strategy injects:

In [ ]:
print("Default code-execution prefix")
print("-" * 60)
print(_DEFAULT_PREFIX)
print("-" * 60)
print(f"name:        {CodeExecStrategy().name!r}")
print(f"description: {CodeExecStrategy().description}")

Now run the strategy with a `MockModel` that simulates an LLM
writing a script. The mock returns a tool call to `execute_python`
with the code; the loop runs it; the mock then reads the structured
result on its next turn and produces a final answer.

In [ ]:
from arcrun._messages import system_message, user_message
from arcrun.events import EventBus
from arcrun.registry import ToolRegistry
from arcrun.sandbox import Sandbox
from arcrun.state import RunState

exec_tool = make_execute_tool(timeout_seconds=5)

code_model = MockModel(
    [
        LLMResponse(
            content="I'll compute the sum.",
            tool_calls=[
                ToolCall(
                    id="tc1",
                    name="execute_python",
                    arguments={"code": "print(sum(range(101)))"},
                )
            ],
            stop_reason="tool_use",
        ),
        LLMResponse(content="The sum of 0..100 is 5050.", stop_reason="end_turn"),
    ]
)

bus = EventBus(run_id="demo-code")
registry = ToolRegistry(tools=[exec_tool], event_bus=bus)
state = RunState(
    messages=[
        system_message("You are helpful."),
        user_message("What is the sum of 0 to 100?"),
    ],
    registry=registry,
    event_bus=bus,
    run_id="demo-code",
    strategy_name="code",
)
sandbox = Sandbox(config=None, event_bus=bus)

result = await CodeExecStrategy()(code_model, state, sandbox, max_turns=5)

print(f"content:        {result.content}")
print(f"strategy_used:  {result.strategy_used}")
print(f"turns:          {result.turns}")
print(f"tools called:   {result.tool_calls_made}")

Inspect the augmented system message — exactly what the LLM saw on
its first turn. The original prompt is preserved at the end; the
code-execution prefix is layered on top.

In [ ]:
augmented = state.messages[0].content
print(f"length:                  {len(augmented)} chars")
print(f"starts with code prefix: {augmented.startswith('You have access to')}")
print(f"original at tail:        {'You are helpful.' in augmented}")

events = [e for e in result.events if e.type == "code.prompt.augmented"]
print(f"\ncode.prompt.augmented event: {events[0].data}")

**Custom prefix**: pass `system_prompt_prefix` to override the
default. Useful when you want different guidance for a specific
agent (e.g., "always use NumPy", "prefer pure stdlib").

In [ ]:
custom_strat = CodeExecStrategy(system_prompt_prefix="Always use list comprehensions.")

bus2 = EventBus(run_id="demo-custom")
reg2 = ToolRegistry(tools=[exec_tool], event_bus=bus2)
state2 = RunState(
    messages=[system_message("Original."), user_message("Compute.")],
    registry=reg2,
    event_bus=bus2,
    run_id="demo-custom",
    strategy_name="code",
)

await custom_strat(
    MockModel([LLMResponse(content="Done.", stop_reason="end_turn")]),
    state2,
    Sandbox(config=None, event_bus=bus2),
    max_turns=3,
)
print(repr(state2.messages[0].content))

## 4. `make_execute_tool` — local sandboxed Python

The default backend runs scripts in a Python subprocess. The
factory returns a `Tool` ready to register. Key isolation
guarantees:

- **New process group** (`start_new_session=True`) so the loop can
  signal the whole tree at once on timeout.
- **Minimal env** — only `PATH=/usr/bin:/bin`, `HOME=/tmp`, `LANG`.
  Your real `HOME` is **not** leaked. Pass `extra_env` to add more.
- **Fresh temp dir** as `cwd` for every execution. Cleaned up after.
- **Two-phase shutdown** on timeout: `SIGTERM`, 5s grace, then
  `SIGKILL` to the process group.
- **Output truncation** at `max_output_bytes` (default 64 KiB).
- **Structured JSON return** — `{stdout, stderr, exit_code,
  duration_ms}` so the LLM sees a uniform schema.

What you get is *enough isolation for trusted environments* — the
host filesystem and network are still reachable. For untrusted
input, use `make_contained_execute_tool` (section 5).

In [ ]:
tool = make_execute_tool(timeout_seconds=10)
print(f"name:            {tool.name}")
print(f"description:     {tool.description}")
print(f"timeout_seconds: {tool.timeout_seconds}  (None = managed internally)")
print(f"input_schema:    {tool.input_schema}")

Direct execution — call `tool.execute({...}, ctx)` to run a script
outside the loop. Useful for unit-testing the sandbox itself.

In [ ]:
import asyncio
import json

from arcrun.types import ToolContext

ctx = ToolContext(
    run_id="demo",
    tool_call_id="tc-demo",
    turn_number=1,
    event_bus=None,
    cancelled=asyncio.Event(),
)

raw = await tool.execute({"code": "print(2 + 2)"}, ctx)
print(json.dumps(json.loads(raw), indent=2))

**Failure cases** are first-class. The script raises a Python
exception → `exit_code != 0`, traceback in `stderr`. The LLM sees
this as ordinary tool feedback and can iterate.

In [ ]:
raw = await tool.execute({"code": "raise ValueError('bad')"}, ctx)
result = json.loads(raw)
print(f"exit_code:   {result['exit_code']}")
print(f"stderr tail: ...{result['stderr'].strip()[-60:]}")

**Env isolation** — verify `HOME` is not your real home, and `cwd`
is a temp dir. The next two cells assert both.

In [ ]:
raw = await tool.execute({"code": "import os; print(os.environ.get('HOME', 'NOT SET'))"}, ctx)
print("HOME inside subprocess:", json.loads(raw)["stdout"].strip())
print("(real HOME is /Users/...; here it's /tmp)")

In [ ]:
raw = await tool.execute({"code": "import os; print(os.getcwd())"}, ctx)
print("cwd:", json.loads(raw)["stdout"].strip())
print("(temp dir, removed after execution)")

**Two-phase timeout** — a 1-second timeout on a 10-second sleep
produces `exit_code = -1` and `Error: execution timed out`. The
subprocess is killed via `SIGTERM` (grace) then `SIGKILL` to the
process group, so child processes are cleaned up too.

In [ ]:
fast = make_execute_tool(timeout_seconds=1)
raw = await fast.execute({"code": "import time; time.sleep(10); print('never')"}, ctx)
result = json.loads(raw)
print(f"exit_code: {result['exit_code']}")
print(f"stderr:    {result['stderr']!r}")
print(f"stdout:    {result['stdout']!r}  (empty — never reached print)")

**Output truncation** — when stdout exceeds `max_output_bytes`,
the captured payload is hard-clipped. Prevents a runaway script
from blowing the LLM's context window.

In [ ]:
small = make_execute_tool(max_output_bytes=50)
raw = await small.execute({"code": "print('x' * 1000)"}, ctx)
result = json.loads(raw)
print(f"stdout length: {len(result['stdout'])} (max=50)")
assert len(result["stdout"]) <= 50

**Custom env** — `extra_env` is merged on top of the default
minimal env so you can inject specific variables (e.g.
`PYTHONPATH`) without leaking the host environment.

In [ ]:
with_env = make_execute_tool(extra_env={"MY_VAR": "injected"})
raw = await with_env.execute({"code": "import os; print(os.environ.get('MY_VAR'))"}, ctx)
print("MY_VAR:", json.loads(raw)["stdout"].strip())

## 5. `make_contained_execute_tool` — Docker isolation

When you need stronger isolation than a subprocess — untrusted
input, federal compliance, resource ceilings — use the container
backend. Defaults are aggressively locked down:

| Setting | Default | Why |
|---|---|---|
| `mem_limit` | `"256m"` | Cap RAM. OOM kill (exit 137) → `SandboxOOMError`. |
| `cpu_period` / `cpu_quota` | 100k / 50k | 50% of one CPU core. |
| `pids_limit` | `64` | Stop fork bombs. |
| `tmpfs_size` | `"64m"` | `/tmp` is in-memory, `noexec`, `nosuid`. |
| `network_disabled` | `True` | No outbound traffic. |
| `read_only` | `True` | Root FS is read-only. |
| `cap_drop` | `["ALL"]` | No Linux capabilities. |
| `security_opt` | `["no-new-privileges"]` | No setuid escalation. |
| `user` | `65534:65534` (`nobody`) | Unprivileged inside the container. |

Code is shipped in via `put_archive` (a tar stream) — never as a
shell argument, so quoting bugs cannot become injection bugs. The
image is auto-detected from `DOCKER_HOST` and falls back through
`/run/user/$UID/podman/podman.sock` and `/var/run/docker.sock`. A
warning is logged if the image is not pinned by digest
(`@sha256:...`).

**Public API**: the factory lives in
`arcrun.builtins.contained_execute`. The error hierarchy is
re-exported at the package level — error types are importable
even when the docker SDK is not installed.

In [ ]:
from arcrun import (
    SandboxError,
    SandboxOOMError,
    SandboxRuntimeError,
    SandboxTimeoutError,
    SandboxUnavailableError,
)
from arcrun.builtins.contained_execute import make_contained_execute_tool

for cls in (
    SandboxError,
    SandboxUnavailableError,
    SandboxTimeoutError,
    SandboxOOMError,
    SandboxRuntimeError,
):
    print(f"{cls.__name__:28s}  bases={[b.__name__ for b in cls.__bases__]}")

**Calling the factory** without a running container runtime would
raise `SandboxUnavailableError` at first execution. To show the
API works without forcing the demo to start a real container, we
first probe Docker availability. If the daemon is up, we run a
tiny container live; if not, we print the configured tool surface
and skip the live invocation.

In [ ]:
import shutil
import subprocess


def docker_available() -> bool:
    """True iff a Docker/Podman daemon is reachable right now."""
    if shutil.which("docker") is None and shutil.which("podman") is None:
        return False
    cli = "docker" if shutil.which("docker") else "podman"
    try:
        out = subprocess.run(  # noqa: S603 - cli is whitelisted to docker/podman above
            [cli, "info"],
            capture_output=True,
            timeout=3,
            check=False,
        )
    except (subprocess.TimeoutExpired, OSError):
        return False
    return out.returncode == 0


DOCKER_AVAILABLE = docker_available()
print(f"Docker daemon reachable: {DOCKER_AVAILABLE}")

**Inspecting the configured tool** (no container started yet) —
the factory needs the `docker` Python SDK installed but the daemon
is only contacted at execute time. So we can build the tool and
look at its `Tool` surface regardless of whether the daemon is
running.

In [ ]:
try:
    contained = make_contained_execute_tool(
        image="python:3.11-slim",
        timeout_seconds=10,
        mem_limit="128m",
        cpu_quota=25_000,  # 25% of one core
        pids_limit=32,
        network_disabled=True,
        read_only=True,
    )
    print(f"name:        {contained.name}")
    print(f"description: {contained.description}")
    print(f"schema:      {contained.input_schema}")
    print("Tool built — limits configured: mem=128m, cpu=25%, pids=32, no network, RO FS")
except ImportError as exc:
    print(f"docker SDK not installed: {exc}")

**Live container demo** — gated on Docker availability. When the
daemon is up, this pulls `python:3.11-slim` (if not already
cached), starts a locked-down container, ships the script in via
tar, runs it, captures stdout, and tears the container down. When
the daemon is down, we print a skip message — the smoke run still
passes.

In [ ]:
if DOCKER_AVAILABLE:
    try:
        live_tool = make_contained_execute_tool(
            image="python:3.11-slim",
            timeout_seconds=30,
        )
        raw = await live_tool.execute(
            {"code": "print('hello from inside the container')"},
            ctx,
        )
        result = json.loads(raw)
        print("live container output:")
        print(f"  stdout:      {result['stdout'].strip()!r}")
        print(f"  exit_code:   {result['exit_code']}")
        print(f"  duration_ms: {result['duration_ms']}")
    except (SandboxError, ImportError) as exc:
        # ImportError: the daemon/CLI is reachable but the `docker` Python SDK
        # isn't installed in this environment (pip install arcrun[container]).
        print(f"Container run failed ({type(exc).__name__}): {exc}")
else:
    print("Docker not available -- skipping live container demo (real API shown above)")

**Mocked container exec** — for CI and offline development we
verify the factory's lockdown defaults by patching the docker SDK.
This is exactly how the package's own test suite covers container
behaviour without requiring a live daemon (see
`packages/arcrun/tests/test_contained_execute.py`).

In [ ]:
import sys
from unittest.mock import MagicMock, patch


def _make_mock_docker(stdout=b"hello", stderr=b"", exit_code=0):
    mock_docker = MagicMock()
    mock_container = MagicMock()
    exec_result = MagicMock()
    exec_result.exit_code = exit_code
    exec_result.output = (stdout, stderr)
    mock_container.exec_run.return_value = exec_result
    mock_docker.DockerClient.return_value.containers.create.return_value = mock_container
    return mock_docker, mock_container


mock_docker, mock_container = _make_mock_docker(stdout=b"42\n")
with patch.dict(sys.modules, {"docker": mock_docker}):
    with patch("pathlib.Path.exists", return_value=True):
        tool = make_contained_execute_tool(image="python:3.11-slim")
        raw = await tool.execute({"code": "print(42)"}, ctx)
        out = json.loads(raw)
        kwargs = mock_docker.DockerClient.return_value.containers.create.call_args.kwargs

print("Mocked container result:")
print(f"  stdout:    {out['stdout']!r}")
print(f"  exit_code: {out['exit_code']}")

print("\nLockdown config passed to containers.create:")
for key in (
    "user",
    "mem_limit",
    "cpu_period",
    "cpu_quota",
    "pids_limit",
    "network_disabled",
    "read_only",
    "cap_drop",
    "security_opt",
    "tmpfs",
):
    print(f"  {key:18s} {kwargs.get(key)!r}")

**OOM handling** — when the container is killed by the kernel for
exceeding `mem_limit`, Docker exits with code 137. The factory
translates that into `SandboxOOMError` so callers can distinguish
resource exhaustion from ordinary script failure.

In [ ]:
mock_docker, _ = _make_mock_docker(exit_code=137)
with patch.dict(sys.modules, {"docker": mock_docker}):
    with patch("pathlib.Path.exists", return_value=True):
        tool = make_contained_execute_tool(image="python:3.11-slim")
        try:
            await tool.execute({"code": "x = 'a' * 10**10"}, ctx)
        except SandboxOOMError as exc:
            print(f"caught: {type(exc).__name__}: {exc}")

**Code-size cap** — the factory enforces a 1 MiB ceiling on the
script payload. Anything bigger raises `SandboxRuntimeError`
before a container is even started.

In [ ]:
from arcrun.builtins.contained_execute import MAX_CODE_BYTES

mock_docker = MagicMock()
with patch.dict(sys.modules, {"docker": mock_docker}):
    with patch("pathlib.Path.exists", return_value=True):
        tool = make_contained_execute_tool(image="python:3.11-slim")
        big = "x = 1\n" * 200_000
        try:
            await tool.execute({"code": big}, ctx)
        except SandboxRuntimeError as exc:
            print(f"caught: {type(exc).__name__}: {exc}")

print(f"MAX_CODE_BYTES = {MAX_CODE_BYTES:,}")

## 6. `prompt_guidance` — what each strategy advertises

Every `Strategy` subclass exposes three properties:

- `name` — canonical identifier (`"react"`, `"code"`)
- `description` — one-liner used in strategy-selection prompts
- `prompt_guidance` — multi-paragraph instructions for the LLM
  about *when and how to use this strategy*

The ABC enforces all three as abstract — a strategy that forgets
to define `prompt_guidance` cannot be instantiated. This is how
ArcRun keeps strategy documentation co-located with strategy code
while still letting consuming agents (ArcAgent etc.) inject the
text into their own system prompts.

In [ ]:
react = ReactStrategy()
code = CodeExecStrategy()

for s in (react, code):
    print(f"=== {s.name.upper()} ===")
    print(s.prompt_guidance)
    print()

**Try to define a Strategy without `prompt_guidance`** — the ABC
rejects it. This is the contract that keeps the system prompt
honest as new strategies land.

In [ ]:
class IncompleteStrategy(Strategy):
    @property
    def name(self) -> str:
        return "incomplete"

    @property
    def description(self) -> str:
        return "missing prompt_guidance"

    async def __call__(self, model, state, sandbox, max_turns): ...


try:
    IncompleteStrategy()
except TypeError as exc:
    print(f"TypeError: {exc}")

## 7. `get_strategy_prompts()` — assembling model-facing strategy text

The consuming agent (e.g. `arcagent.ArcAgent`) needs to assemble a
system prompt that teaches the LLM about ArcRun's strategies and
any arcrun-owned tools (`execute_python`,
`contained_execute_python`). It calls `get_strategy_prompts` with:

- `allowed_strategies` — the names that will be available
- `tool_names` — names of tools that will be registered

and gets back a dict keyed by section name. The agent decides the
ordering and concatenation; ArcRun supplies the *text*.

In [ ]:
from arcrun.prompts import CODE_EXEC_GUIDANCE, CONTAINED_EXEC_GUIDANCE

default = get_strategy_prompts()
print("keys when called with no args:")
for k in default:
    print(f"  - {k}")

**Tool-aware**: when you pass `tool_names`, the function detects
ArcRun-owned execution tools and includes their guidance. Tools
owned by other packages (e.g. `spawn_task` from arcagent) are
intentionally a no-op here — their guidance lives with their
owner.

In [ ]:
with_exec = get_strategy_prompts(tool_names=["execute_python"])
print("with execute_python tool:")
for k in with_exec:
    print(f"  - {k}")

with_both = get_strategy_prompts(
    tool_names=["execute_python", "contained_execute_python", "spawn_task"]
)
print("\nwith both exec tools + spawn_task:")
for k in with_both:
    print(f"  - {k}")
print("(no spawn_guidance -- arcrun does not own spawn_task)")

**Multi-strategy mode**: passing two or more strategies adds a
`strategy_selection` section that summarises each option for the
model. This is what gets injected when an agent allows the model
to choose between ReAct and CodeExec.

In [ ]:
multi = get_strategy_prompts(
    allowed_strategies=["react", "code"],
    tool_names=["execute_python"],
)
for k in multi:
    print(f"--- {k} ---")
    print(multi[k][:200])
    print()

**The two arcrun-owned constants** (`CODE_EXEC_GUIDANCE`,
`CONTAINED_EXEC_GUIDANCE`) are public so agents can also import
them directly when assembling custom prompts.

In [ ]:
print("CODE_EXEC_GUIDANCE first 200 chars:")
print(CODE_EXEC_GUIDANCE[:200])
print("...\n")
print("CONTAINED_EXEC_GUIDANCE first 200 chars:")
print(CONTAINED_EXEC_GUIDANCE[:200])

## 8. Strategy selection — the dispatch mechanism

When `run()` (or `run_async()`) is called with `allowed_strategies`,
ArcRun's `select_strategy` decides which one runs. The logic in
`packages/arcrun/src/arcrun/strategies/__init__.py` has three
paths:

| `allowed_strategies` | What happens | LLM calls |
|---|---|---|
| `None` | Default to `"react"` | 0 |
| `["code"]` | Use that strategy directly | 0 |
| `["react", "code"]` | Model picks via `select_strategy` tool | 1 |

Selection uses a real LLM tool call with an `enum` constraint, so
the response is guaranteed to be a valid strategy name. Any
exception or invalid choice falls back to `"react"` and emits
`strategy.selection.fallback` (or `.error`) for audit.

In [ ]:
# Path 1: None -> default to react, no LLM call
bus = EventBus(run_id="sel-none")
reg = ToolRegistry(tools=[exec_tool], event_bus=bus)
state = RunState(
    messages=[user_message("hi")],
    registry=reg,
    event_bus=bus,
    run_id="sel-none",
)
chosen = await select_strategy(None, model=None, state=state)
print(f"None         -> {chosen!r}")

In [ ]:
# Path 2: single-element list -> use directly, no LLM call
bus = EventBus(run_id="sel-single")
reg = ToolRegistry(tools=[exec_tool], event_bus=bus)
state = RunState(
    messages=[user_message("hi")],
    registry=reg,
    event_bus=bus,
    run_id="sel-single",
)
chosen = await select_strategy(["code"], model=None, state=state)
print(f"['code']     -> {chosen!r}")

In [ ]:
# Path 3: model picks. Show what the model is shown, then verify event emission.
bus = EventBus(run_id="sel-multi")
reg = ToolRegistry(tools=[exec_tool], event_bus=bus)
state = RunState(
    messages=[user_message("compute the 30th fibonacci number")],
    registry=reg,
    event_bus=bus,
    run_id="sel-multi",
)

selector = MockModel(
    [
        LLMResponse(
            tool_calls=[
                ToolCall(
                    id="sel1",
                    name="select_strategy",
                    arguments={
                        "strategy": "code",
                        "reasoning": "computational task -- code is more direct",
                    },
                )
            ],
            stop_reason="tool_use",
        )
    ]
)

chosen = await select_strategy(["react", "code"], selector, state)
print(f"['react','code'] -> {chosen!r}\n")

print("events emitted during selection:")
for e in bus.events:
    print(f"  {e.type:34s} {dict(e.data)}")

**What the selector model is shown**: the system message lists
every strategy and its description, the registered tools, and the
task. The selector is given a single `select_strategy` tool whose
schema enums to the allowed names — the model literally cannot
return an invalid strategy.

In [ ]:
call = selector.invoke_calls[0]
system_msg = call["messages"][0].content
user_msg = call["messages"][1].content
print("system:")
print(system_msg)
print(f"\nuser:  {user_msg!r}")
print(
    f"\nselect_strategy schema enum: {call['tools'][0].parameters['properties']['strategy']['enum']}"
)

**Fallback paths** keep the system fail-safe. Invalid strategy
name → fall back to `react`. Model raises an exception → fall
back to `react`. Both emit dedicated events so a forensic review
can detect when selection drifted.

In [ ]:
bus = EventBus(run_id="sel-bad-name")
reg = ToolRegistry(tools=[exec_tool], event_bus=bus)
state = RunState(
    messages=[user_message("x")],
    registry=reg,
    event_bus=bus,
    run_id="sel-bad-name",
)
bad = MockModel(
    [
        LLMResponse(
            tool_calls=[
                ToolCall(id="sel1", name="select_strategy", arguments={"strategy": "made_up"})
            ],
            stop_reason="tool_use",
        )
    ]
)

chosen = await select_strategy(["react", "code"], bad, state)
print(f"invalid name -> {chosen!r}")
fb = [e for e in bus.events if e.type == "strategy.selection.fallback"]
print(f"fallback event: {dict(fb[0].data)}")

In [ ]:
class ExplodingModel:
    async def invoke(self, messages, tools=None, **_):
        raise RuntimeError("upstream is down")


bus = EventBus(run_id="sel-error")
reg = ToolRegistry(tools=[exec_tool], event_bus=bus)
state = RunState(
    messages=[user_message("x")],
    registry=reg,
    event_bus=bus,
    run_id="sel-error",
)

chosen = await select_strategy(["react", "code"], ExplodingModel(), state)
print(f"model exception -> {chosen!r}")
err = [e for e in bus.events if e.type == "strategy.selection.error"]
print(f"error event: {dict(err[0].data)}")

**Unknown strategy in `allowed`**: the function rejects it eagerly
with `ValueError`. This is the only configuration error
`select_strategy` raises — bad runtime input is forgiven, bad
configuration is not.

In [ ]:
bus = EventBus(run_id="sel-bad-config")
reg = ToolRegistry(tools=[exec_tool], event_bus=bus)
state = RunState(
    messages=[user_message("x")],
    registry=reg,
    event_bus=bus,
    run_id="sel-bad-config",
)

try:
    await select_strategy(["react", "unknown"], MockModel([]), state)
except ValueError as exc:
    print(f"ValueError: {exc}")

## 9. Mid-run `task_complete` — early termination from inside CodeExec

CodeExec is just `react_loop` with a different prompt, which means
every loop-control tool works inside it. The most useful is
`task_complete` — a builtin that signals structured completion
with a `status` (`success`, `partial`, `failed`), a `summary`, and
optional artifacts. The loop watches for any tool flagged
`signals_completion=True` and terminates immediately.

Full deep-dive: `walkthroughs/arcrun/06-task-completion-budgets.ipynb`.
Here we just verify it composes with CodeExec.

In [ ]:
from arcrun.builtins import make_task_complete_tool

tc_tool = make_task_complete_tool()
print(f"name:                 {tc_tool.name}")
print(f"signals_completion:   {tc_tool.signals_completion}")
print(f"description:          {tc_tool.description}")

Run a CodeExec session that does one execution, then calls
`task_complete` with a structured payload. The loop terminates
after the completion tool fires, regardless of how many turns
remain in `max_turns`.

In [ ]:
model = MockModel(
    [
        LLMResponse(
            tool_calls=[
                ToolCall(
                    id="tc1",
                    name="execute_python",
                    arguments={"code": "print(7 * 6)"},
                )
            ],
            stop_reason="tool_use",
        ),
        LLMResponse(
            tool_calls=[
                ToolCall(
                    id="done",
                    name="task_complete",
                    arguments={
                        "status": "success",
                        "summary": "7 * 6 = 42",
                        "artifacts": ["answer=42"],
                    },
                )
            ],
            stop_reason="tool_use",
        ),
    ]
)

bus = EventBus(run_id="demo-tc")
reg = ToolRegistry(tools=[exec_tool, tc_tool], event_bus=bus)
state = RunState(
    messages=[system_message("You are helpful."), user_message("What is 7*6?")],
    registry=reg,
    event_bus=bus,
    run_id="demo-tc",
    strategy_name="code",
)

result = await CodeExecStrategy()(model, state, Sandbox(config=None, event_bus=bus), max_turns=10)
print(f"strategy:    {result.strategy_used}")
print(f"turns:       {result.turns}  (max was 10)")
print(f"content:     {result.content!r}")
print()
completed = [e for e in result.events if e.type == "loop.completed"]
print(f"loop.completed payload: {dict(completed[0].data)}")

## 10. End-to-end through `run()`

Tying it all together: `arcrun.run` is the public entry point.
Pass it the model, the tools, an `allowed_strategies` list, and
an optional `on_event` handler. Below we run the full pipeline
twice — once letting the model pick between ReAct and CodeExec
(it picks CodeExec), and once forcing CodeExec directly.

In [ ]:
from arcrun import StaticProvider, run

events_seen: list[str] = []


def on_event(event):
    events_seen.append(event.type)


model = MockModel(
    [
        # Strategy selection turn
        LLMResponse(
            tool_calls=[
                ToolCall(id="sel", name="select_strategy", arguments={"strategy": "code"})
            ],
            stop_reason="tool_use",
        ),
        # CodeExec loop turn 1: write code
        LLMResponse(
            tool_calls=[
                ToolCall(
                    id="tc1",
                    name="execute_python",
                    arguments={"code": "print(sum(i*i for i in range(11)))"},
                )
            ],
            stop_reason="tool_use",
        ),
        # Loop turn 2: final answer
        LLMResponse(content="Sum of squares 0..10 = 385.", stop_reason="end_turn"),
    ]
)

result = await run(
    model=model,
    capabilities=StaticProvider([make_execute_tool(timeout_seconds=5)]),
    system_prompt="You are helpful.",
    task="What is the sum of i*i for i in 0..10?",
    allowed_strategies=["react", "code"],
    on_event=on_event,
)

print(f"strategy_used: {result.strategy_used}")
print(f"content:       {result.content}")
print(f"turns:         {result.turns}")
print()
print("event types in order:")
for et in events_seen:
    print(f"  {et}")

Forcing a single strategy avoids the selection round-trip — useful
when the agent already knows it wants code execution. Note
`strategy.selection.start` is **not** emitted because the loop
skips the LLM-based selection path.

In [ ]:
events_seen.clear()

model = MockModel(
    [
        LLMResponse(
            tool_calls=[
                ToolCall(
                    id="tc1",
                    name="execute_python",
                    arguments={"code": "print('ok')"},
                )
            ],
            stop_reason="tool_use",
        ),
        LLMResponse(content="Ran the script.", stop_reason="end_turn"),
    ]
)

result = await run(
    model=model,
    capabilities=StaticProvider([make_execute_tool(timeout_seconds=5)]),
    system_prompt="You are helpful.",
    task="Run any code.",
    allowed_strategies=["code"],
    on_event=on_event,
)

print(f"strategy_used: {result.strategy_used}")
print("event types:")
for et in events_seen:
    print(f"  {et}")

## 11. Summary

**Strategies are interface, not engine.** Both `ReactStrategy` and
`CodeExecStrategy` delegate to `react_loop`. CodeExec adds a
system-prompt prefix; that is the entire behavioural difference.

**Two execution backends, same tool shape.**
`make_execute_tool` runs scripts in a Python subprocess with a
minimal env, fresh temp dir, and two-phase timeout — fast, good
for trusted local use. `make_contained_execute_tool` runs scripts
in a locked-down Docker container with `mem_limit`, `cpu_quota`,
`pids_limit`, no network, read-only FS, dropped capabilities, and
`nobody` user — slower but safe for untrusted input or
compliance-bound deployments.

**Prompt provider keeps separation clean.**
`prompt_guidance` is an abstract property on `Strategy`. The
package-level `get_strategy_prompts()` assembles the right
fragments for the active strategies and tools — strategy +
selection + arcrun-owned tool guidance — and hands them to the
consuming agent. Other tools' guidance lives with their owners
(arcagent owns `spawn_task` guidance, etc.).

**Strategy selection has three deterministic paths.**
`None` → `react`. Single name → use directly. Multiple → model
picks via a `select_strategy` tool whose schema enums to the
allowed names. Invalid choices and exceptions fall back to
`react` and emit dedicated events so audit can detect drift.

**Mid-run completion still works.** `task_complete` flows through
CodeExec exactly the way it flows through ReAct — the
completion-detection lives in `react_loop`, not in either
strategy. Deep-dive in
`walkthroughs/arcrun/06-task-completion-budgets.ipynb`.

**Public API exercised in this notebook:**
- `arcrun.Strategy` (ABC)
- `arcrun.strategies.STRATEGIES`, `_load_strategies`,
  `select_strategy`
- `arcrun.strategies.code.CodeExecStrategy`, `_DEFAULT_PREFIX`
- `arcrun.strategies.react.ReactStrategy`
- `arcrun.make_execute_tool` (`extra_env`, `timeout_seconds`,
  `max_output_bytes`)
- `arcrun.builtins.contained_execute.make_contained_execute_tool`
  (`mem_limit`, `cpu_quota`, `pids_limit`, `network_disabled`,
  `read_only`, `tmpfs_size`)
- `arcrun.SandboxError`, `SandboxOOMError`, `SandboxTimeoutError`,
  `SandboxRuntimeError`, `SandboxUnavailableError`
- `arcrun.get_strategy_prompts`, `arcrun.prompts.CODE_EXEC_GUIDANCE`,
  `CONTAINED_EXEC_GUIDANCE`
- `arcrun.builtins.make_task_complete_tool`
- `arcrun.run` (with `allowed_strategies`, `on_event`)

**Next:**
- `walkthroughs/arcrun/04-streaming.ipynb` — same loop, real-time
  output via `run_stream`.
- `walkthroughs/arcrun/06-task-completion-budgets.ipynb` —
  `task_complete` deep-dive plus `max_cost_usd` and `max_turns`
  budget enforcement.
- `walkthroughs/arcrun/07-event-chain-verification.ipynb` —
  hash-chained events as a tamper-evident audit trail.